<a href="https://colab.research.google.com/github/sleep-is-best/Ai/blob/main/beta_0.1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
!pip install -q transformers[sentencepiece] protobuf

KeyboardInterrupt: 

In [7]:
!pip install -q transformers[sentencepiece] protobuf opencv-python tqdm

### 🏺 نظام التدريب الشامل للأبجدية الميسينية (Linear B)

In [11]:
import pandas as pd
import torch
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from PIL import Image
import cv2
import os

# تحديث الأبجدية الميسينية الشاملة بناءً على طلب المستخدم
full_mycenaean_mapping = {
    '𐀀': 'a', '𐀁': 'e', '𐀂': 'i', '𐀃': 'o', '𐀄': 'u',
    '𐀅': 'da', '𐀆': 'de', '𐀇': 'di', '𐀈': 'do', '𐀉': 'du',
    '𐀊': 'ja', '𐀋': 'je', '𐀍': 'jo', '𐀎': 'ju',
    '𐀏': 'ka', '𐀐': 'ke', '𐀑': 'ki', '𐀒': 'ko', '𐀓': 'ku',
    '𐀔': 'ma', '𐀕': 'me', '𐀖': 'mi', '𐀗': 'mo', '𐀘': 'mu',
    '𐀙': 'na', '𐀚': 'ne', '𐀛': 'ni', '𐀜': 'no', '𐀝': 'nu',
    '𐀞': 'pa', '𐀟': 'pe', '𐀠': 'pi', '𐀡': 'po', '𐀢': 'pu',
    '𐀣': 'qa', '𐀤': 'qe', '𐀥': 'qi', '𐀦': 'qo',
    '𐀨': 'ra', '𐀩': 're', '𐀪': 'ri', '𐀫': 'ro', '𐀬': 'ru',
    '𐀭': 'sa', '𐀮': 'se', '𐀯': 'si', '𐀰': 'so', '𐀱': 'su',
    '𐀲': 'ta', '𐀳': 'te', '𐀴': 'ti', '𐀵': 'to', '𐀶': 'tu',
    '𐀷': 'wa', '𐀸': 'we', '𐀹': 'wi', '𐀺': 'wo',
    '𐀼': 'za', '𐀽': 'ze', '𐀿': 'zo',
    '𐁀': 'a2', '𐁁': 'a3', '𐁂': 'au', '𐁃': 'dwe', '𐁄': 'dwo',
    '𐁅': 'nwa', '𐁇': 'pte', '𐁆': 'pu2', '𐁈': 'ra2', '𐁉': 'ra3',
    '𐁊': 'ro2', '𐁋': 'ta2', '𐁌': 'twe', '𐁍': 'two'
}

# إضافة الرموز الفاصلة
full_mycenaean_mapping['𐄁'] = 'separator'

data = {
    'file_name': ['/content/sample_data/IMAGES/اثار للغة الميسينية.png'],
    'text': ['𐀗𐀛𐄁𐀀𐀸𐀆𐄁𐀳𐀀𐄁𐀟𐀩𐀷𐀆𐀃𐀍𐄁𐀀𐀑𐀩ဃa𐄁']
}
df_labels = pd.DataFrame(data)
display(df_labels)
print(f"Total Characters mapped: {len(full_mycenaean_mapping)}")

,file_name,text
0,/content/sample_data/IMAGES/اثار للغة الميسيني...,𐀗𐀛𐄁𐀀𐀸𐀆𐄁𐀳𐀀𐄁𐀟𐀩𐀷𐀆𐀃𐀍𐄁𐀀𐀑𐀩ဃa𐄁


Total Characters mapped: 75


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class MycenaeanDataset(Dataset):
    def __init__(self, df, processor):
        self.df = df
        self.processor = processor

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = os.path.join('/content', self.df['file_name'][idx])
        image = cv2.imread(img_path)
        if image is None: raise FileNotFoundError(f"Image not found at {img_path}")

        # المعالجة المسبقة لتحسين التعرف
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        _, thresh = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY_INV)

        pixel_values = self.processor(Image.fromarray(thresh).convert("RGB"), return_tensors="pt").pixel_values
        labels = self.processor.tokenizer(self.df['text'][idx], padding="max_length", max_length=128).input_ids
        labels = [label if label != self.processor.tokenizer.pad_token_id else -100 for label in labels]

        return {"pixel_values": pixel_values.squeeze(), "labels": torch.tensor(labels)}

In [12]:
import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW
from tqdm.auto import tqdm

# وضع النموذج في وضع التدريب
model.train()

# إعداد المحسن والبيانات مع التوسيع (Augmentation)
optimizer = AdamW(model.parameters(), lr=2e-5)
scaler = torch.amp.GradScaler('cuda') if device.type == 'cuda' else None

dataset = AugmentedMycenaeanDataset(df_labels, processor, augment=True)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

epochs = 100
print(f'استئناف التدريب الجاري لـ {epochs} دورة...')

for epoch in range(epochs):
    epoch_loss = 0
    progress_bar = tqdm(dataloader, desc=f'Epoch {epoch+1}/{epochs}', leave=False)

    for batch in progress_bar:
        optimizer.zero_grad()
        if scaler:
            with torch.amp.autocast('cuda'):
                outputs = model(pixel_values=batch['pixel_values'].to(device), labels=batch['labels'].to(device))
                loss = outputs.loss
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(pixel_values=batch['pixel_values'].to(device), labels=batch['labels'].to(device))
            loss = outputs.loss
            loss.backward()
            optimizer.step()

        epoch_loss += loss.item()
        progress_bar.set_postfix({'loss': f'{loss.item():.4f}'})

    if (epoch + 1) % 10 == 0:
        print(f'الدورة {epoch+1}/{epochs} - Loss: {epoch_loss/len(dataloader):.4f}')

print('اكتملت هذه المرحلة من التدريب. يمكنك إكمال العمل لاحقاً.')

استئناف التدريب لـ 100 دورة إضافية مع شريط التقدم...


Epoch 1/100:   0%|          | 0/3 [00:00<?, ?it/s]

Epoch 2/100:   0%|          | 0/3 [00:00<?, ?it/s]

Epoch 3/100:   0%|          | 0/3 [00:00<?, ?it/s]

Epoch 4/100:   0%|          | 0/3 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [10]:
import torchvision.transforms as T
from PIL import Image
import numpy as np

# تعريف تقنيات تكبير البيانات
augmentation_transforms = T.Compose([
    T.RandomRotation(degrees=2),
    T.RandomAffine(degrees=0, translate=(0.02, 0.02), scale=(0.98, 1.02)),
    T.ColorJitter(brightness=0.2, contrast=0.2),
])

class AugmentedMycenaeanDataset(MycenaeanDataset):
    def __init__(self, df, processor, augment=False):
        super().__init__(df, processor)
        self.augment = augment

    def __getitem__(self, idx):
        # جلب البيانات الأساسية من الكلاس الأصلي
        possible_paths = [
            os.path.join('/content/sample_data/IMAGES', self.df['file_name'][idx] + '.jpg'),
            os.path.join('/content/sample_data/IMADES', self.df['file_name'][idx] + '.jpg')
        ]
        image = None
        for path in possible_paths:
            if os.path.exists(path):
                image = Image.open(path).convert('RGB')
                break

        if image is None: raise FileNotFoundError(f'Image not found for {self.df["file_name"][idx]}')

        if self.augment:
            image = augmentation_transforms(image)

        pixel_values = self.processor(image, return_tensors='pt').pixel_values
        labels = self.processor.tokenizer(self.df['text'][idx], padding='max_length', max_length=128).input_ids
        labels = [label if label != self.processor.tokenizer.pad_token_id else -100 for label in labels]

        return {'pixel_values': pixel_values.squeeze(), 'labels': torch.tensor(labels)}

print('تم إعداد نظام تكبير البيانات (Data Augmentation). يمكنك الآن استخدامه في التدريب لزيادة التنوع.')

تم إعداد نظام تكبير البيانات (Data Augmentation). يمكنك الآن استخدامه في التدريب لزيادة التنوع.


In [ ]:
model.eval()
image_path = '/content/اثار للغة الميسينية.png'
image = cv2.imread(image_path)
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
thresh = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 11, 2)
pixel_values = processor(Image.fromarray(thresh).convert('RGB'), return_tensors='pt').pixel_values.to(device)

with torch.no_grad():
    generated_ids = model.generate(
        pixel_values,
        max_new_tokens=50,
        num_beams=5,
        repetition_penalty=3.0,
        length_penalty=1.0,
        early_stopping=True
    )
    generated_text = processor.tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

print(f'Detected text (raw): {generated_text}')

decoded_output = []
for char in generated_text:
    if char in full_mycenaean_mapping:
        decoded_output.append(full_mycenaean_mapping[char])
    else:
        decoded_output.append(f'[{char}]')

print(f'Phonetic Translation: {"-".join(decoded_output)}')

### استخراج النصوص من جميع الصور

الآن، بعد تدريب النموذج، يمكننا استخدامه لاستخراج النصوص من جميع الصور في مجموعة البيانات `df_labels`.

In [6]:
model.eval()

full_mycenaean_mapping = {
    '𐀀': 'a', '𐀁': 'e', '𐀂': 'i', '𐀃': 'o', '𐀄': 'u',
    '𐀅': 'da', '𐀆': 'de', '𐀇': 'di', '𐀈': 'do', '𐀉': 'du',
    '𐀊': 'ja', '𐀋': 'je', '𐀍': 'jo', '𐀎': 'ju',
    '𐀏': 'ka', '𐀐': 'ke', '𐀑': 'ki', '𐀒': 'ko', '𐀓': 'ku',
    '𐀔': 'ma', '𐀕': 'me', '𐀖': 'mi', '𐀗': 'mo', '𐀘': 'mu',
    '𐀙': 'na', '𐀚': 'ne', '𐀛': 'ni', '𐀜': 'no', '𐀝': 'nu',
    '𐀞': 'pa', '𐀟': 'pe', '𐀠': 'pi', '𐀡': 'po', '𐀢': 'pu',
    '𐀣': 'qa', '𐀤': 'qe', '𐀥': 'qi', '𐀦': 'qo',
    '𐀨': 'ra', '𐀩': 're', '𐀪': 'ri', '𐀫': 'ro', '𐀬': 'ru',
    '𐀭': 'sa', '𐀮': 'se', '𐀯': 'si', '𐀰': 'so', '𐀱': 'su',
    '𐀲': 'ta', '𐀳': 'te', '𐀴': 'ti', '𐀵': 'to', '𐀶': 'tu',
    '𐀷': 'wa', '𐀸': 'we', '𐀹': 'wi', '𐀺': 'wo',
    '𐀼': 'za', '𐀽': 'ze', '𐀿': 'zo',
    '𐁀': 'a2', '𐁁': 'a3', '𐁂': 'au', '𐁃': 'dwe', '𐁄': 'dwo',
    '𐁅': 'nwa', '𐁇': 'pte', '𐁆': 'pu2', '𐁈': 'ra2', '𐁉': 'ra3',
    '𐁊': 'ro2', '𐁋': 'ta2', '𐁌': 'twe', '𐁍': 'two',
    '𐄁': 'separator', 'I': '1', 'II': '2'
}

image_directories = ['/content/sample_data/IMAGES', '/content/sample_data/IMADES']

print("\n--- استخراج النصوص والترجمة الصوتية ---")
for index, row in df_labels.iterrows():
    file_name = row['file_name']
    image = None
    for directory in image_directories:
        image_path = os.path.join(directory, file_name + '.jpg')
        if os.path.exists(image_path):
            image = cv2.imread(image_path)
            break

    if image is None: continue

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    thresh = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 11, 2)
    pixel_values = processor(Image.fromarray(thresh).convert('RGB'), return_tensors='pt').pixel_values.to(device)

    with torch.no_grad():
        generated_ids = model.generate(
            pixel_values,
            max_new_tokens=30,
            num_beams=3,
            repetition_penalty=1.5,
            length_penalty=1.0,
            early_stopping=True
        )
        generated_text = processor.tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

    decoded = [full_mycenaean_mapping.get(c, f'[{c}]') for c in generated_text]

    print(f"\n--- صورة {file_name} ---")
    print(f"النص المستخرج (Linear B): {generated_text}")
    print(f"النطق الصوتي: {'-'.join(decoded)}")


--- استخراج النصوص والترجمة الصوتية ---

--- صورة 1 ---
النص المستخرج: 𐀕𐀁𐀵𐀏
النطق الصوتي: me-e-to-ka

--- صورة 2 ---
النص المستخرج: 𐀏𐀁𐀕𐀒𐀿𐀇𐀵𐀷𐀲𐀜𐀅𐀥𐀊𐀆𐀁𐀁𐀁𐀁𐀁𐀁𐀁𐀁𐀁𐀁𐀁𐀁𐀁𐀁𐀁
النطق الصوتي: ka-e-me-ko-zo-di-to-wa-ta-no-da-qi-ja-de-e-e-e-e-e-e-e-e-e-e-e-e-e-e-e

--- صورة 3 ---
النص المستخرج: 𐀒𐀁𐀕𐀏𐀿𐀇𐀵𐀷𐀥𐀺𐀂𐀆𐀯𐀞𐀁𐀁𐀁𐀁𐀁𐀁𐀁𐀁𐀁𐀁𐀁𐀁𐀁𐀁𐀁
النطق الصوتي: ko-e-me-ka-zo-di-to-wa-qi-wo-i-de-si-pa-e-e-e-e-e-e-e-e-e-e-e-e-e-e-e

--- صورة 4 ---
النص المستخرج: 𐀏𐀁𐀕𐀒𐀥𐀵
النطق الصوتي: ka-e-me-ko-qi-to

--- صورة 5 ---
النص المستخرج: 𐀵𐀁𐀕𐀏𐀒
النطق الصوتي: to-e-me-ka-ko

--- صورة 6 ---
النص المستخرج: 𐀵𐀁𐀕IIIIII𐀥I𐀺IIIIIIIIIIIIIIIIIIII
النطق الصوتي: to-e-me-1-1-1-1-1-1-qi-1-wo-1-1-1-1-1-1-1-1-1-1-1-1-1-1-1-1-1-1-1-1


In [8]:
import shutil
from google.colab import files

# حفظ النموذج والمعالج
model.save_pretrained('./mycenaean_trocr_model')
processor.save_pretrained('./mycenaean_trocr_model')

# ضغط المجلد لتسهيل تحميله
shutil.make_archive('mycenaean_model', 'zip', './mycenaean_trocr_model')

print("تم حفظ النموذج بنجاح في مجلد المحتوى (content).")
# تم تعطيل سطر التحميل بناءً على طلبك
# files.download('mycenaean_model.zip')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

KeyboardInterrupt: 

### 📝 النصوص الأصلية للبيانات (Ground Truth)

| الصورة | النص الميسيني (Linear B) |
|---|---|
| 1 | `𐀆𐀵𐀕𐀵𐀡` |
| 2 | `𐀏𐀨𐀁𐀪𐀥𐀈𐀜𐀇𐀏𐀲𐀆𐀸𐀥` |
| 3 | `𐀞𐀂𐁖𐀅𐀲𐀒𐀷𐀕𐀿𐀁𐀇𐀅𐀏𐀒𐀺𐀕𐀁𐀇` |
| 4 | `𐀢𐀠𐀊𐀰𐀷𐀕𐀿𐀁𐀕𐀿𐀁` |
| 5 | `𐀛𐀑𐀍𐀒𐀜𐀯𐀊𐄸𐄹` |
| 6 | `𐀵𐀁𐀐𐀮𐀯𐀞I𐀳𐀺𐀂𐀥II𐀀𐀩` |

In [9]:
import os

def get_size(start_path = '.'):
    total_size = 0
    for dirpath, dirnames, filenames in os.walk(start_path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            total_size += os.path.getsize(fp)
    return total_size

model_dir_size = get_size('./mycenaean_trocr_model') / (1024 * 1024)
zip_file_size = os.path.getsize('mycenaean_model.zip') / (1024 * 1024)

print(f"حجم مجلد النموذج: {model_dir_size:.2f} MB")
print(f"حجم الملف المضغوط (للتنزيل): {zip_file_size:.2f} MB")

حجم مجلد النموذج: 1277.58 MB
حجم الملف المضغوط (للتنزيل): 385.41 MB
